# 19.19 下載到權重，怎樣確認自己用對版本？


同名模型經過再次訓練，答案與Auto規則都可能改變。因此一份可使用的成品，要把**權重、網路設定、tokenizer及決策規則**對到同一個固定版本。下載網址只告訴我們去哪裡取得檔案，檔案指紋才協助我們確認拿到的是哪份內容。

新成品用兩層清單配對版本：release.json固定 Hugging Face（HF）上的 commit、成品目錄及 manifest 指紋；各成品的 artifact manifest 再列推論檔案的大小與SHA-256。推論時先核對，再建立網路、載入權重、切成評估模式；評估模式關閉某些訓練行為，但不會把抽樣自動變成固定答案。讀出方式、輪數及裝置仍須一起記。首次下載成功與真正離線載入推論，是兩項不同檢查。

三個成品各有獨立的推論指令：MoE 助理沿用[19.11](https://birdhackor.github.io/tiny-perceptron-vlm/19.11.html)的公開CPU操作；Looped 解題器接受有限文字與輪數；機率判斷器接受狀態、問題與候選。Auto 指令同時載入解題器、判斷器與凍結的決策規則，並接受 effort 設定。使用本課已訓練完成的權重作推論，不代表你正在訓練，也不代表借用了外部成熟神經模型。

若要自己重訓，先用小設定確定資料與程式能運作，再選CPU、Apple裝置的GPU運算後端MPS（Metal Performance Shaders），或NVIDIA GPU。完整作者訓練可用GPU，學生不必在筆電重跑全部正式實驗。訓練配方另記seed、更新步數、資料版本、檢查點選擇與硬體；可續訓的檢查點還要有最佳化器與隨機狀態，不能只用推論權重假裝無縫續訓。

程式碼與生成器保存在Git；大量資料、檢查點與推論權重放在既有的資料／模型儲存位置。公開檔案應有來源與授權，不能把私人存取金鑰（access token）或帶憑證的下載連結寫入教材。下載與上傳驗證會保留可公開的檔案hash及結果，避免把環境祕密當操作教學。

以下命令在自己電腦的終端機執行，不是 Notebook 程式格：沿用[W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1)的CPU環境，在專案根目錄啟用`.venv`後執行。`--product b` 是 Looped 解題器、`--product c` 是機率判斷器；`download`第一次連線取得兩者，後面的命令只用本機檔案：

```bash
python -m tiny_perceptron.lesson_products download --product b
python -m tiny_perceptron.lesson_products download --product c
python -m tiny_perceptron.lesson_products solve --state '兩盒追蹤;A=2;B=6;事件=交換;查詢=B' --loops 3
python -m tiny_perceptron.lesson_products judge --state '卡片進度;申請編號=?' --question info
python -m tiny_perceptron.lesson_products auto --state '兩盒追蹤;A=2;B=6;事件=交換;查詢=B' --effort auto
```

例中的交換把A原來的2移到B，所以正確答案是2；卡片進度缺申請編號，資訊充分性的正確類別是`no`。`--question` 可選 `choice`（服務選擇）、`info`（資訊充分）或 `rubric`（規準等級）。這些短輸入讓你看懂介面，模型在其他有限輸入上仍可能答錯。解題器輸出答案與實際輪數，判斷器輸出候選機率與集中度；Auto 另記實際使用的兩次判斷和解題輪數。在 Auto 指令指定`--effort medium`要求3輪，`--max-loops 2`則會截到2並標`budget_limited`。

[成品操作](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/README.md)列出固定HF版本、候選ID、三種問題、本機JSON服務及重訓命令。可用`serve --port 8765`在本機 127.0.0.1 開一個 HTTP 服務：HTTP 是網頁與程式之間傳送請求的協定，這個服務讓別的程式用網址送出同樣的 solve、judge、auto 請求；它接受有限語法，不是一般聊天服務。完整訓練使用自己的新輸出目錄。

這兩個新成品的推論權重叫 `model.pt`，和 MoE 助理的 `model.safetensors` 是不同成品的不同格式。`model.pt` 是 PyTorch 自己的檔案格式，這裡只存模型參數，程式用 `torch.load(…, weights_only=True)` 讀取：只還原張量與字典這類基本資料，不執行檔案裡任意物件的載入程式；它和[19.11](https://birdhackor.github.io/tiny-perceptron-vlm/19.11.html)的公開 safetensors 一樣只供推論，只下載它無法恢復原本的最佳化器與隨機狀態。

練習：只換了解題器的權重，仍用舊的 Auto 規則，為什麼需要重驗？選擇器原本根據舊解題器各檔的表現制定，換權重後成功與成本關係可能改變。版本鎖定保護的是整個決策流程，不只是能否讀檔。
